# 🚀 Model Deployment and Inference with KServe

This notebook demonstrates how to take a trained **Hugging Face** model and transition it into a production-ready inference service. We move beyond local experimentation to show how a model can be hosted as a scalable microservice within the **Kubeflow** ecosystem.

### 🎯 Objectives
* **Programmatic Deployment:** Use the **KServe Python SDK** to define and launch an `InferenceService` (ISVC) using the Hugging Face serving runtime.
* **Infrastructure as Code:** Configure hardware resource limits (CPU/Memory) and service annotations to ensure stable model hosting.
* **Service Validation:** Utilize the `KServeClient` to poll the deployment status and confirm when the model is ready to receive traffic.
* **High-Performance Inference:** Construct and send **V2 Inference Protocol** (OpenInference) HTTP requests to the model endpoint.
* **Result Parsing:** Process the JSON response to extract predictions (in this case, for a `fill-mask` task) and validate the model's output logic.

---

### 🛠️ Technical Stack
* **KServe SDK:** For lifecycle management of the inference service.
* **Hugging Face Runtime:** A specialized server container optimized for transformer models.
* **V2 Inference Protocol:** The industry-standard REST API format for high-performance model serving.

In [ ]:
!pip install -r requirements.txt

In [ ]:
ISVC_NAME="llm-model"

## Deployment 

In this section, we leverage the KServe Python SDK to programmatically define an InferenceService, which abstracts away the complexity of managing the underlying Kubernetes pods, autoscaling, and networking. By specifying the huggingface model format and providing a storageUri, KServe automatically provisions an optimized runtime environment tailored for transformer models.

In [ ]:
from kserve import KServeClient
from kserve import V1beta1InferenceService
from kserve import V1beta1InferenceServiceSpec
from kserve import V1beta1PredictorSpec
from kserve import V1beta1ModelSpec, V1beta1ModelFormat
from kubernetes import client as k8s_client

# 1. Initialize the KServe Client
# If running inside a Kubeflow notebook, it automatically picks up credentials
kserve_client = KServeClient()

# 2. Define the InferenceService
isvc = V1beta1InferenceService(
    api_version="serving.kserve.io/v1beta1",
    kind="InferenceService",
    metadata=k8s_client.V1ObjectMeta(
        name=ISVC_NAME,
        annotations={'sidecar.istio.io/inject': 'false'}
    ),
    spec=V1beta1InferenceServiceSpec(
        predictor=V1beta1PredictorSpec(
            model=V1beta1ModelSpec(
                model_format=V1beta1ModelFormat(name="huggingface"),
                storage_uri="hf://sshleifer/tiny-distilroberta-base",
                args=[
                    "--backend=huggingface",
                    "--task=fill_mask"
                ],
                resources=k8s_client.V1ResourceRequirements(
                    requests={"cpu": "100m", "memory": "600Mi"},
                    limits={"cpu": "1", "memory": "1Gi"}
                )
            )
        )
    )
)

# 3. Create the InferenceService
kserve_client.create(isvc)

print(f"Deployment of {ISVC_NAME} initiated.")

In [ ]:
from tenacity import retry, wait_exponential, stop_after_attempt

# Retry logic to ensure the Inference Service is ready
@retry(
    wait=wait_exponential(multiplier=2, min=1, max=10),
    stop=stop_after_attempt(30),
    reraise=True,
)
def assert_isvc_created(client, isvc_name):
    assert client.is_isvc_ready(isvc_name), f"Failed to create Inference Service {isvc_name}."

assert_isvc_created(kserve_client, ISVC_NAME)

## Inference

Now that the model is up and running, let's try to query it with some data

In [ ]:
# Initialize the KServe client
# This client is used to interact with the KServe Inference Service.
kserve_client = KServeClient()

# Retrieve the Inference Service details
# Fetches the Inference Service by name and extracts the URL for making predictions.
isvc_resp = kserve_client.get(ISVC_NAME)
inference_service_url = isvc_resp["status"]["address"]["url"]
print("Inference URL:", inference_service_url)

In [ ]:
import requests

# The URL remains the same (ending in /infer)
url = f"{inference_service_url}/v2/models/{ISVC_NAME}/infer"

payload = {
    "inputs": [
        {
            "name": "args", # The HuggingFace runtime usually expects 'args' or 'input'
            "shape": [2],   # We are sending 2 strings
            "datatype": "BYTES",
            "data": [
                "HuggingFace is a <mask>-based company.",
                "The model will <mask> the missing word."
            ]
        }
    ]
}

response = requests.post(url, json=payload)
response.json()